# 후쿠오카 숙박업소 리뷰 감성분석 대시보드

## 실행 순서
1. **[1단계]** 패키지 설치
2. **[2단계]** 파일 업로드 (`app.py` + `final_result_v3.csv`)
3. **[3단계]** Streamlit 실행 → 링크 클릭

> GPU 불필요 — 런타임 유형은 기본(CPU)으로 두셔도 됩니다.

## [1단계] 패키지 설치
2~3분 소요됩니다.

In [ ]:
!pip install -q streamlit pandas plotly wordcloud matplotlib
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('✅ 패키지 설치 완료')

## [2단계] 기존 파일 정리 + 업로드
이전 세션 파일을 삭제하고 새로 업로드합니다.

In [ ]:
import os, glob
for f in glob.glob('/content/*.csv') + glob.glob('/content/app.py'):
    os.remove(f)
    print(f'삭제: {f}')
print('✅ 정리 완료')

In [ ]:
from google.colab import files
print('app.py 와 final_result_v3.csv 를 함께 선택해서 업로드해주세요.')
uploaded = files.upload()
print('\n업로드된 파일:', list(uploaded.keys()))

import glob as g
def find_file(pattern):
    matches = g.glob(f'/content/{pattern}')
    if not matches:
        raise FileNotFoundError(f'{pattern} 파일을 찾을 수 없습니다.')
    return sorted(matches)[0]

APP_FILE = find_file('app.py')
CSV_FILE = find_file('final_result_v3*.csv')
print(f'\n앱 파일: {APP_FILE}')
print(f'데이터: {CSV_FILE}')
print('✅ 파일 확인 완료 — [3단계]를 실행하세요.')

## [3단계] Streamlit 실행
셀을 실행하면 `trycloudflare.com` 링크가 출력됩니다. 클릭하면 대시보드가 열립니다.

> - 링크가 나오기까지 20~30초 걸립니다
> - 셀이 `[*]` 상태여야 대시보드가 유지됩니다
> - 대시보드 사이드바에서 `final_result_v3.csv`를 업로드하면 분석이 시작됩니다

In [ ]:
import subprocess, time, re

PORT = 8501

# Streamlit 실행
streamlit_proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port', str(PORT),
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false',
     '--server.maxUploadSize', '200'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

print('Streamlit 기동 중... (10초 대기)')
time.sleep(10)

# Cloudflare Tunnel 연결
print('Cloudflare Tunnel 연결 중...')
cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
)

# URL 추출
url = None
for _ in range(30):
    line = cf_proc.stderr.readline()
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        break

if url:
    print('\n' + '='*60)
    print('✅ 대시보드 링크 (클릭하세요):')
    print(f'   {url}')
    print('='*60)
    print('링크 클릭 후 사이드바에서 final_result_v3.csv 를 업로드하세요.')
    print('이 셀이 실행 중([*])인 동안 대시보드가 유지됩니다.')
else:
    print('⚠️  URL을 가져오지 못했습니다. 셀을 다시 실행해주세요.')

try:
    cf_proc.wait()
except KeyboardInterrupt:
    streamlit_proc.terminate()
    cf_proc.terminate()
    print('대시보드 종료')

---
## 대시보드 탭 구성

| 탭 | 내용 |
|---|---|
| 📊 개요 | 전체 감성 비율, 평점 분포, 프로젝트 파이프라인 |
| ☁️ 워드클라우드 | 전체·긍정·부정 핵심 단어 시각화 |
| 🔍 RQ1 | 평점-감성 불일치 분석, 불일치 케이스 목록 |
| 🏘️ RQ2 | 숙소 유형·업소별 비교, 업소별 워드클라우드 |
| 📈 RQ3 | 연도별 트렌드, 코로나 전후 비교 |
| 📝 RQ4 | 텍스트 길이·플랫폼 보조 분석 |
| 🤖 모델 성능 비교 | v1/v2/v3 F1 비교, 파인튜닝 효과 |
| 📋 데이터 | 필터링 + CSV 다운로드 |

### 오류 대처
| 오류 | 해결 |
|---|---|
| 링크가 안 열림 | 30초 기다린 후 새로고침 |
| `ModuleNotFoundError` | [1단계] 셀을 다시 실행 |
| 파일명 충돌 | [2단계] 첫 번째 셀(파일 삭제)을 먼저 실행 후 업로드 |